In [1]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())


2.2.2


In [2]:
!ls /pool/data/ERA5/E5/

E5.bib	ml  pl	sf


In [3]:
import os
import xarray as xr
import pandas as pd
from cdo import Cdo
from joblib import Parallel, delayed

# Paths
day_path_sf  = "/pool/data/ERA5/E5/sf/fc/1D/"

scratch_path = "/scratch/u/u301827/paris/"
final_path   = "/work/uc1275/u301827/02_MSE/paris/raw/"

os.makedirs(scratch_path, exist_ok=True)
os.makedirs(final_path,   exist_ok=True)

# Paris coordinates
PARIS_LON = 2.35
PARIS_LAT = 48.86
lon_min = lon_max = PARIS_LON
lat_min = lat_max = PARIS_LAT


In [4]:
def process_era5_paris(year, month, var_num, var,
                       levels="ml", level="137"):
    """
    Process ERA5 daily data selecting ONLY the Paris grid cell.
    
    Notes:
    - ERA5 daily (1D) files in the pool are already aggregated from hourly data,
      so no daily aggregation is necessary here.
    - This function extracts the Paris grid cell and applies vertical level
      selection where required (model or pressure levels).
    """

    cdo = Cdo()
    date_str = f"{year}-{month:02d}"

    # Output folders
    var_scratch = os.path.join(scratch_path, var)
    var_final   = os.path.join(final_path, var)
    os.makedirs(var_scratch, exist_ok=True)
    os.makedirs(var_final,   exist_ok=True)

    out_file = os.path.join(var_final, f"{var}_{date_str}_paris.nc")

    # ---------------------------------------------------------
    # Determine which ERA5 path to use
    # ---------------------------------------------------------
    if levels == "ml":
        data_path = data_path_ml
    elif levels == "pl":
        data_path = data_path_pl
    else:
        data_path = day_path_sf   # surface daily fields

    var_file = f"{data_path}{var_num}/E5{levels}12_1D_{date_str}_{var_num}.grb"
    if not os.path.exists(var_file):
        print(f"Missing: {var_file}")
        return

    # Step 1: Convert GRIB → NetCDF + regular grid
    reg_file = os.path.join(var_scratch, f"reg_{var}_{date_str}.nc")
    cdo.setgridtype("regular", input=var_file, output=reg_file, options="-f nc --eccodes")

    # Step 2: Select only the Paris grid cell
    paris_file = os.path.join(var_scratch, f"paris_{var}_{date_str}.nc")
    # Step 2: select Paris grid cell (nearest grid point)
    cdo.remapnn(f"lon={PARIS_LON}_lat={PARIS_LAT}", input=reg_file, output=paris_file)

    # ---------------------------------------------------------
    # Step 3: Apply vertical level logic
    # ---------------------------------------------------------

    # Model levels (ML)
    if levels == "ml":
        cdo.sellevel(level, input=paris_file, output=out_file)
        os.remove(paris_file)

    # Pressure levels (PL)
    elif levels == "pl":
        ds = xr.open_dataset(paris_file)
        ds_sel = ds.sel(plev=int(level))
        ds_sel["time"] = pd.to_datetime(ds_sel["time"].values)
        ds_sel.to_netcdf(out_file)
        ds.close()
        os.remove(paris_file)

    # Surface fields (SF)
    else:
        # Surface fields (SF)
        import shutil
        shutil.move(paris_file, out_file)

    # Clean up temp file
    os.remove(reg_file)

    return out_file


In [ ]:
# === Variable mapping ===
era5_vars = {
    59: "cape",
    159: "blh",
    182: "e",
    507: "pev",
    146: "sshf",
    147: "slhf",
    49: "10fg"
}

# === Date range (monthly) ===
start_date = "1940-01-01"
end_date   = "2024-12-31"
months = pd.date_range(start_date, end_date, freq="MS")  # monthly start dates

# === Wrapper function for correct level handling ===
def run_process(date, var_num, var):

    # Pressure-level variables (500 hPa)
    if var_num in [130, 129]:
        return process_era5_paris(
            year=date.year,
            month=date.month,
            var_num=f"{var_num:03d}",
            var=var,
            levels="pl",
            level="50000"
        )

    # Model-level variable (q at ML 137)
    elif var_num == 133:
        return process_era5_paris(
            year=date.year,
            month=date.month,
            var_num=f"{var_num:03d}",
            var=var,
            levels="ml",
            level="137"
        )

    # Surface variable (2m temperature)
    elif var_num in [59, 159, 507, 182, 146, 147,49]:
        return process_era5_paris(
            year=date.year,
            month=date.month,
            var_num=f"{var_num:03d}",
            var=var,
            levels="sf"
        )

    else:
        print(f"Variable {var_num} not supported.")
        return None

# === Parallel execution ===
from joblib import Parallel, delayed

n_jobs = 50

results = Parallel(n_jobs=n_jobs, verbose=10)(
    delayed(run_process)(month, var_num, var)
    for var_num, var in era5_vars.items()
    for month in months
)

print("Processing completed.")


[Parallel(n_jobs=50)]: Using backend LokyBackend with 50 concurrent workers.
[Parallel(n_jobs=50)]: Done  13 tasks      | elapsed:    6.7s
[Parallel(n_jobs=50)]: Done  28 tasks      | elapsed:    7.3s
[Parallel(n_jobs=50)]: Done  45 tasks      | elapsed:    7.6s
[Parallel(n_jobs=50)]: Done  62 tasks      | elapsed:   10.5s
[Parallel(n_jobs=50)]: Done  81 tasks      | elapsed:   11.0s
[Parallel(n_jobs=50)]: Done 100 tasks      | elapsed:   12.1s
[Parallel(n_jobs=50)]: Done 121 tasks      | elapsed:   14.4s
[Parallel(n_jobs=50)]: Done 142 tasks      | elapsed:   15.4s
[Parallel(n_jobs=50)]: Done 165 tasks      | elapsed:   18.0s
[Parallel(n_jobs=50)]: Done 188 tasks      | elapsed:   18.8s
[Parallel(n_jobs=50)]: Done 213 tasks      | elapsed:   21.4s
[Parallel(n_jobs=50)]: Done 238 tasks      | elapsed:   22.5s
[Parallel(n_jobs=50)]: Done 265 tasks      | elapsed:   25.1s
[Parallel(n_jobs=50)]: Done 292 tasks      | elapsed:   26.4s
[Parallel(n_jobs=50)]: Done 321 tasks      | elapsed:  

In [7]:
import os
import pandas as pd
from joblib import Parallel, delayed

# === Example variable mapping ===
# We'll just test one variable first, e.g., 2m temperature (surface)
era5_vars = {
    167: "2t",  # 2m temperature, surface (no levels)
}

# === Date range for testing ===
start_date = "2021-06-01"
end_date   = "2021-06-01"
months = pd.date_range(start=start_date, end=end_date, freq="MS")

# === Wrapper to call process_era5_paris correctly ===
def run_process(month, var_num, var):
    """
    Calls process_era5_paris for a single month and variable,
    automatically setting levels depending on the variable.
    """
    # Pressure-level variables
    if var_num in [130, 129]:
        return process_era5_paris(
            year=month.year, month=month.month,
            var_num=str(var_num), var=var,
            levels="pl", level="50000"
        )

    # Model-level variables
    elif var_num == 133:
        return process_era5_paris(
            year=month.year, month=month.month,
            var_num=str(var_num), var=var,
            levels="ml", level="137"
        )

    # Surface variables (2m temperature)
    elif var_num == 167:
        return process_era5_paris(
            year=month.year, month=month.month,
            var_num=str(var_num), var=var,
            levels="sf"
        )

    else:
        print(f"Variable {var_num} not configured.")
        return None

# === Test execution for a single month ===
for var_num, var in era5_vars.items():
    for month in months:
        out_file = run_process(month, var_num, var)
        if out_file:
            print(f"Processed {var} for {month.strftime('%Y-%m')}: {out_file}")
        else:
            print(f"Failed to process {var} for {month.strftime('%Y-%m')}")


Processed 2t for 2021-06: /work/uc1275/u301827/02_MSE/paris/raw/2t/2t_2021-06_paris.nc


## 